In [1]:
%pip install "Faker==40.23.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
from faker import Faker
import pandas as pd
import random
import json
from datetime import time

In [33]:
df_pacientes = pd.read_csv("./datos/turno",header=0)

In [34]:
cuils = df_pacientes['cuil_paciente'].sample(frac=0.05, random_state=42) # tomamos 500 de los cuils que tuvieron turnos para que realicen encuestas


In [35]:
data_gen = Faker('es_AR')
seed = 14422 
random.seed(seed)
Faker.seed(seed)

In [36]:
malos_comentarios = ["Mala experiencia","Demasiada demora, mal trato", "No volveria","Muy disconforme","Desagradable"]
buenos_comentarios = ["Muy veloz","Buena experiencia", "Genial trato","El médico respondió todas mis dudas.","La recepción fue muy amable.","Excelente atención."]

In [37]:
with open ('./datos/encuestas.json','w') as f:
    for cuil in cuils:
        encuesta = {
            "cuil" : str(cuil),
            "puntajes" : {"atencion_medica": random.randint(1, 10),
                        "recepcion": random.randint(1, 10),
                        "limpieza": random.randint(1, 10)},
            "fecha": data_gen.date_between(start_date="-2y", end_date="today").isoformat() 
        }
        promedio = sum(encuesta["puntajes"].values())/3
        if promedio < 4 or (promedio < 6 and random.random() < 0.5):
            encuesta["recomienda"] = False
            encuesta["demora"] = random.randint(30,150) 
        else:
            encuesta["recomienda"] = True
            encuesta["demora"] = random.randint(0,45) 


        if random.random() < 0.3:
            if not encuesta["recomienda"]:
                encuesta["comentario"] = random.choice(malos_comentarios)
            else:
                encuesta["comentario"] = random.choice(buenos_comentarios)
        json.dump(encuesta,f)
        f.write("\n")

In [38]:
random.seed(seed)
Faker.seed(seed)

In [39]:
tipo_notif = ["recordatorio_turno", "confirmacion_turno", "cancelacion_turno", "reprogramacion_turno", "resultado_estudio", "campania_prevencion", "encuesta_satisfaccion"]
canal = ["email", "sms", "app","pagina web"]
estado = ["pendiente", "enviado", "fallido", "leido"]
motivo = ["error de conexion", "usuario inaccesible","buzon lleno","tiempo de espera agotado","servidor no disponible"]


In [40]:
with open ('./datos/notif.json','w') as f:
    for cuil in cuils:
        notif ={
            "cuil" : str(cuil),
            "tipo" : random.choice(tipo_notif),
            "canal" : random.choice(canal),
            "intentos" :{"estado1" : random.choice(estado)},
            "fecha": data_gen.date_between(start_date="-2y", end_date="today").isoformat() 
        }
        leido = False
        if notif["intentos"]["estado1"] == "leido": # si lo leyó primer intento
            notif["intentos"]["cant_lecturas"] = random.randint(1,3)
            leido = True
        elif notif["intentos"]["estado1"] == "fallido": # si fallo primer intento
            notif["intentos"]["motivo1"] = random.choice(motivo)
            notif["intentos"]["estado2"] = random.choice(estado)

            if notif["intentos"]["estado2"] == "leido": # si lo leyó segundo intento
                notif["intentos"]["cant_lecturas"] = random.randint(1,3)
                leido = True
            elif notif["intentos"]["estado2"] == "fallido": # si fallo segundo intento
                notif["intentos"]["motivo2"] = random.choice(motivo)
                notif["intentos"]["estado3"]= random.choice(estado)

                if notif["intentos"]["estado3"] == "leido": # si lo leyó tercer intento
                    notif["intentos"]["cant_lecturas"] = random.randint(1,3)
                    leido = True
                elif notif["intentos"]["estado3"] == "fallido": # si fallo tercer intento
                   notif["intentos"]["motivo3"] = random.choice(motivo) 
        if notif["tipo"] == "encuesta_satisfaccion":
            if leido:
                notif["encuesta_realizada"] = True
            else:
                notif["encuesta_realizada"] = False
        json.dump(notif,f)
        f.write("\n")